# Projet : Stage

## 1. Hiérarchisation des données

### 1.1. Arrondissement/Cantons/Communes/Departements/Regions

In [1]:
import json

In [2]:
def fillChamp(dicChamps, dic_hierarchisee, rang) :

    for i in range(len(dicChamps['features'])) :
        champ = dicChamps['features'][i]['properties']['nom']
        if champ not in dic_hierarchisee[rang] :
            dic_hierarchisee[rang].append(champ)
        
    return 

def fillDictionnaire() :
    fichiers = ['arrondissements', 'cantons', 'communes', 'departements', 'regions']
    dic_hierarchisee = {}

    for fichier in fichiers :
        dic_hierarchisee[fichier] = []
        mon_json = open(f"Education/levels/france-geojson/{fichier}-avec-outre-mer.geojson")
        data = json.load(mon_json)
        mon_json.close()

        fillChamp(data, dic_hierarchisee, fichier)

    return dic_hierarchisee

In [3]:
dic_hierarchisee = fillDictionnaire()

### 1.2 Quartiers

In [4]:
import csv

In [5]:

fichier = open("Education/levels/liste-correspondance-qp2024-qp2015.csv", "r", encoding="utf-8")
reader = csv.reader(fichier, delimiter=";")
listeQuartiers = list(reader)[1:]

In [6]:
dic_hierarchisee['quartiers'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][1]
    if quartier not in dic_hierarchisee['quartiers'] :
        dic_hierarchisee['quartiers'].append(quartier)

dic_hierarchisee['QP'] = []
for i in range(len(listeQuartiers)) :
    quartier = listeQuartiers[i][3]
    if quartier not in dic_hierarchisee['QP'] :
        dic_hierarchisee['QP'].append(quartier)

del i, listeQuartiers, quartier, fichier, reader

In [7]:
for key in dic_hierarchisee:
    dic_hierarchisee[key].sort()

## 2. Identification des attributs spatiaux dans un fichier csv/xlsx

### 2.1 Algorithme de recherche

Mon objectif est de parcourir un tableau en vérifiant à chaque cellule si elle appartient à une donnée de mon dictionnaire jusqu'à trouver la plus petite et la plus grande granularité

In [23]:
# Tableau rangé par granularité des champs
champs = ['QP', 'quartiers', 'arrondissements', 'cantons', 'communes', 'departements', 'regions']
dic_hierarchisee = {champ: dic_hierarchisee[champ] for champ in champs if champ in dic_hierarchisee}

In [20]:
fichier = open("Education/csv/fr-esr-insersup.csv", encoding="utf-8")
reader = csv.reader(fichier, delimiter=";")
listeChamps = list(reader)[1:]
fichier.close()

In [ ]:
attributs_trouves = []
for i in range(10) :
    for j in range (len(listeChamps[0])) :
        for champ, valeur in dic_hierarchisee.items() :
            if listeChamps[i][j] in valeur :
                if champ not in attributs_trouves :
                    attributs_trouves.append(champ)
del i, j, champ, valeur, listeChamps, fichier, reader
attributs_trouves

['regions', 'arrondissements', 'communes', 'QP']

In [ ]:
def recherchePremiereOccurence(champs, attributs_trouves) :
    for i in range(len(champs)) :
        for j in range (len(champs[0])) :
            for champ in attributs_trouves :
                if champs[i] in dic_hierarchisee[champ] :
                    return i, j, champ
    return


'regions'

### 2.2 Fichiers csv